# Fake News Detection & Verification Platform

## Model Training & Comparison

This notebook trains and compares machine learning models for fake-news
classification using TF-IDF features generated from the WELFake dataset.

### Dataset

Final WELFake dataset:

- Total articles: 63,050
- Fake: 34,788
- Real: 28,262

### Models

1. Logistic Regression
2. Multinomial Naive Bayes

### Evaluation Metrics

- Accuracy
- Precision
- Recall
- F1 Score
- Training Time
- Prediction Time

### Experimental Principle

All models are trained and evaluated using the same fixed
Train/Validation/Test split and the same TF-IDF representation.

The test set is not used during model selection.

In [ ]:
import pandas as pd
import numpy as np

import pickle
import os
import time

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

In [ ]:
df = pd.read_csv(
    "WELFake_Cleanedfinal.csv"
)

In [ ]:
print(df.shape)
print(df.columns.tolist())

(63050, 5)
['Unnamed: 0', 'title', 'text', 'label', 'combined_text']


In [ ]:
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

In [ ]:
print(df.columns.tolist())

['title', 'text', 'label', 'combined_text']


In [ ]:
print("=" * 50)
print("FINAL DATASET VERIFICATION")
print("=" * 50)

print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:")
print(df.duplicated().sum())

print("\nLabels:")
print(df["label"].value_counts())

print("\nLabel percentages:")
print(
    df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

FINAL DATASET VERIFICATION
Shape: (63050, 4)

Missing values:
title            519
text               0
label              0
combined_text      0
dtype: int64

Duplicates:
0

Labels:
label
0    34788
1    28262
Name: count, dtype: int64

Label percentages:
label
0    55.18
1    44.82
Name: proportion, dtype: float64


In [ ]:
X = df["combined_text"]
y = df["label"]

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [ ]:
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print(
    "Total:",
    len(X_train) +
    len(X_val) +
    len(X_test)
)

Training: 50440
Validation: 6305
Testing: 6305
Total: 63050


In [ ]:
print("TRAIN")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nVALIDATION")
print(y_val.value_counts(normalize=True).mul(100).round(2))

print("\nTEST")
print(y_test.value_counts(normalize=True).mul(100).round(2))

TRAIN
label
0    55.17
1    44.83
Name: proportion, dtype: float64

VALIDATION
label
0    55.18
1    44.82
Name: proportion, dtype: float64

TEST
label
0    55.18
1    44.82
Name: proportion, dtype: float64


In [ ]:
import re

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
nltk.download("omw-1.4")

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
def preprocess_text(text):
    text = str(text)

    text = text.lower()

    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    text = re.sub(
        r"[^a-z\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    tokens = text.split()

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [ ]:
X_train_processed = X_train.apply(preprocess_text)

X_val_processed = X_val.apply(preprocess_text)

X_test_processed = X_test.apply(preprocess_text)

In [ ]:
print(X_train_processed.head())

28881    ammon bundy lunatic militia men facing charge ...
43623    november daily contrarian read november daily ...
55540    obama clock boy come back texas spending month...
18083    president obama make fun donald trump hilariou...
39834    mexico prison population drop police prosecuto...
Name: combined_text, dtype: object


In [ ]:
with open(
    "vectorizer.pkl",
    "rb"
) as file:

    tfidf = pickle.load(file)

In [ ]:
print(tfidf)

TfidfVectorizer(max_df=0.95, max_features=60000, min_df=2, ngram_range=(1, 2),
                sublinear_tf=True)


In [ ]:
print(
    "Vocabulary size:",
    len(tfidf.vocabulary_)
)

Vocabulary size: 60000


In [ ]:
X_train_tfidf = tfidf.transform(
    X_train_processed
)

X_val_tfidf = tfidf.transform(
    X_val_processed
)

X_test_tfidf = tfidf.transform(
    X_test_processed
)

In [ ]:
print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_tfidf.shape)

Train: (50440, 60000)
Validation: (6305, 60000)
Test: (6305, 60000)


In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [ ]:
start_time = time.time()

lr_model.fit(
    X_train_tfidf,
    y_train
)

lr_training_time = time.time() - start_time

print(
    f"Logistic Regression training time: "
    f"{lr_training_time:.2f} seconds"
)

Logistic Regression training time: 4.79 seconds


In [ ]:
start_time = time.time()

lr_val_pred = lr_model.predict(
    X_val_tfidf
)

lr_val_prediction_time = (
    time.time() - start_time
)

In [ ]:
lr_val_accuracy = accuracy_score(
    y_val,
    lr_val_pred
)

lr_val_precision = precision_score(
    y_val,
    lr_val_pred
)

lr_val_recall = recall_score(
    y_val,
    lr_val_pred
)

lr_val_f1 = f1_score(
    y_val,
    lr_val_pred
)

In [ ]:
print("Logistic Regression — Validation")

print(
    "Accuracy:",
    round(lr_val_accuracy, 4)
)

print(
    "Precision:",
    round(lr_val_precision, 4)
)

print(
    "Recall:",
    round(lr_val_recall, 4)
)

print(
    "F1:",
    round(lr_val_f1, 4)
)

print(
    "Prediction time:",
    round(lr_val_prediction_time, 4),
    "seconds"
)

Logistic Regression — Validation
Accuracy: 0.9605
Precision: 0.961
Recall: 0.9505
F1: 0.9557
Prediction time: 0.0064 seconds


In [ ]:
nb_model = MultinomialNB()

In [ ]:
start_time = time.time()

nb_model.fit(
    X_train_tfidf,
    y_train
)

nb_training_time = time.time() - start_time

print(
    f"Naive Bayes training time: "
    f"{nb_training_time:.2f} seconds"
)

Naive Bayes training time: 0.16 seconds


In [ ]:
start_time = time.time()

nb_val_pred = nb_model.predict(
    X_val_tfidf
)

nb_val_prediction_time = (
    time.time() - start_time
)

In [ ]:
nb_val_accuracy = accuracy_score(
    y_val,
    nb_val_pred
)

nb_val_precision = precision_score(
    y_val,
    nb_val_pred
)

nb_val_recall = recall_score(
    y_val,
    nb_val_pred
)

nb_val_f1 = f1_score(
    y_val,
    nb_val_pred
)

In [ ]:
print("Multinomial Naive Bayes — Validation")

print(
    "Accuracy:",
    round(nb_val_accuracy, 4)
)

print(
    "Precision:",
    round(nb_val_precision, 4)
)

print(
    "Recall:",
    round(nb_val_recall, 4)
)

print(
    "F1:",
    round(nb_val_f1, 4)
)

print(
    "Prediction time:",
    round(nb_val_prediction_time, 4),
    "seconds"
)

Multinomial Naive Bayes — Validation
Accuracy: 0.8853
Precision: 0.8575
Recall: 0.8924
F1: 0.8746
Prediction time: 0.0136 seconds


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Multinomial Naive Bayes"
    ],

    "Accuracy": [
        lr_val_accuracy,
        nb_val_accuracy
    ],

    "Precision": [
        lr_val_precision,
        nb_val_precision
    ],

    "Recall": [
        lr_val_recall,
        nb_val_recall
    ],

    "F1": [
        lr_val_f1,
        nb_val_f1
    ],

    "Training Time (s)": [
        lr_training_time,
        nb_training_time
    ],

    "Prediction Time (s)": [
        lr_val_prediction_time,
        nb_val_prediction_time
    ]
})

results

,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Logistic Regression,0.960508,0.961002,0.950460,0.955702,4.790035,0.006392
1,Multinomial Naive Bayes,0.885329,0.857531,0.892427,0.874632,0.159100,0.013609


In [ ]:
results.round(4)

,Model,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Logistic Regression,0.9605,0.9610,0.9505,0.9557,4.7900,0.0064
1,Multinomial Naive Bayes,0.8853,0.8575,0.8924,0.8746,0.1591,0.0136


In [ ]:
best_model_name = results.loc[
    results["F1"].idxmax(),
    "Model"
]

print(
    "Best validation model:",
    best_model_name
)

Best validation model: Logistic Regression


In [ ]:
lr_test_pred = lr_model.predict(
    X_test_tfidf
)

In [ ]:
nb_test_pred = nb_model.predict(
    X_test_tfidf
)

In [ ]:
lr_test_results = {
    "Accuracy": accuracy_score(y_test, lr_test_pred),
    "Precision": precision_score(y_test, lr_test_pred),
    "Recall": recall_score(y_test, lr_test_pred),
    "F1": f1_score(y_test, lr_test_pred)
}

nb_test_results = {
    "Accuracy": accuracy_score(y_test, nb_test_pred),
    "Precision": precision_score(y_test, nb_test_pred),
    "Recall": recall_score(y_test, nb_test_pred),
    "F1": f1_score(y_test, nb_test_pred)
}

In [ ]:
print("Logistic Regression — Test")
print(
    pd.Series(lr_test_results).round(4)
)

print("\nNaive Bayes — Test")
print(
    pd.Series(nb_test_results).round(4)
)

Logistic Regression — Test
Accuracy     0.9583
Precision    0.9543
Recall       0.9526
F1           0.9534
dtype: float64

Naive Bayes — Test
Accuracy     0.8871
Precision    0.8581
Recall       0.8963
F1           0.8768
dtype: float64


In [ ]:
print(
    classification_report(
        y_test,
        lr_test_pred,
        target_names=["Fake", "Real"]
    )
)

              precision    recall  f1-score   support

        Fake       0.96      0.96      0.96      3479
        Real       0.95      0.95      0.95      2826

    accuracy                           0.96      6305
   macro avg       0.96      0.96      0.96      6305
weighted avg       0.96      0.96      0.96      6305



In [ ]:
print(
    classification_report(
        y_test,
        nb_test_pred,
        target_names=["Fake", "Real"]
    )
)

              precision    recall  f1-score   support

        Fake       0.91      0.88      0.90      3479
        Real       0.86      0.90      0.88      2826

    accuracy                           0.89      6305
   macro avg       0.89      0.89      0.89      6305
weighted avg       0.89      0.89      0.89      6305



In [ ]:
os.makedirs(
    "models",
    exist_ok=True
)

In [ ]:
with open(
    "models/logistic_regression.pkl",
    "wb"
) as file:

    pickle.dump(
        lr_model,
        file
    )

In [ ]:
with open(
    "models/naive_bayes.pkl",
    "wb"
) as file:

    pickle.dump(
        nb_model,
        file
    )